# Loading the data

In [56]:
import pandas as pd
import numpy as np

In [57]:
path='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
data=pd.read_csv(path)

In [58]:
data_selected=data[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
data_selected.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


 # Question 1:
* There's one column with missing values. What is it?



In [59]:
data_selected.isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

The column horsepower has 877 missing values. 

# Question 2
* What's the median (50% percentile) for variable 'horsepower'?

In [60]:
med=data_selected['horsepower'].median()
med

np.float64(254.0)

The median value for the variable 'horsepower' is 254.0. 

Splitting the data into train, validation, and test sets

In [61]:
n = len(data_selected)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

data_selected_train = data_selected.iloc[idx[:n_train]]
data_selected_val = data_selected.iloc[idx[n_train:n_train + n_val]]
data_selected_test = data_selected.iloc[idx[n_train + n_val:]]

# Question 3
* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
* Which option gives better RMSE?


In [62]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [63]:
def train_model(train_data,val_data,features,target):
    model=LinearRegression()
    model.fit(train_data[features],train_data[target])
    y_predict=model.predict(val_data[features])
    rmse_empty=root_mean_squared_error(val_data[target],y_predict)
    return rmse_empty

In [64]:
features=['engine_displacement','horsepower','vehicle_weight','model_year']
target='fuel_efficiency_mpg'

In [65]:
data_train_empty=data_selected_train.fillna(0)
data_val_empty=data_selected_val.fillna(0)
rmse_empty=train_model(data_train_empty,data_val_empty,features,target)


In [66]:
data_train_mean=data_selected_train.fillna(data_selected_train['horsepower'].mean())
data_val_mean=data_selected_val.fillna(data_selected_train['horsepower'].mean())
rmse_mean=train_model(data_train_mean,data_val_mean,features,target)

In [67]:
rmse_empty=round(rmse_empty,3)
rmse_mean=round(rmse_mean,3)
print(rmse_empty,rmse_mean)

2.205 2.202


The RMSE for the model is better when we fill the missing values with mean rather than 0.

# Question 4
 - Now let's train a regularized linear regression.
 - For this question, fill the NAs with 0.
 - Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
 - Use RMSE to evaluate the model on the validation dataset.
 - Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie
.- Which r gives the best RMSE?
 - If multiple options give the same best RMSE, select the smallest r.



In [68]:
from sklearn.linear_model import Ridge
ridges = [0, 0.01, 0.1, 1, 5, 10, 100]
for r in ridges:
    model = Ridge(alpha=r)
    model.fit(data_train_empty[features], data_train_empty[target])
    y_predict = model.predict(data_val_empty[features])
    rmse = root_mean_squared_error(data_val_empty[target], y_predict)
    print(f"r: {r}, RMSE: {round(rmse, 4)}")

r: 0, RMSE: 2.2053
r: 0.01, RMSE: 2.2053
r: 0.1, RMSE: 2.2053
r: 1, RMSE: 2.2053
r: 5, RMSE: 2.2053
r: 10, RMSE: 2.2053
r: 100, RMSE: 2.2053


r = 0 gives the best RMSE score of 2.2053

# Question 5
- We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
- Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
- For each seed, do the train/validation/test split with 60%/20%/20% distribution.
- Fill the missing values with 0 and train a model without regularization.
- For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
- What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
- Round the result to 3 decimal digits (round(std, 3))
- What's the value of std?



In [69]:
from sklearn.model_selection import train_test_split

In [70]:
def calc(seed,data_selected,features,target):
    rmse_scores = []
    data_selected_train, data_selected_val = train_test_split(data_selected, test_size=0.2, random_state=seed)
    data_selected_train, data_selected_test = train_test_split(data_selected_train, test_size=0.25, random_state=seed )
    data_selected_train_filled = data_selected_train.fillna(0)
    data_selected_test_filled= data_selected_test.fillna(0)
    data_selected_val_filled = data_selected_val.fillna(0)
    model = LinearRegression()
    model.fit(data_selected_train_filled[features], data_selected_train_filled[target])
    y_predict = model.predict(data_selected_val_filled[features])
    rmse = root_mean_squared_error(data_selected_val_filled[target], y_predict)
    rmse_scores.append(rmse)
    return rmse_scores


In [71]:
test_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
for seed in test_seeds:
    rmse_scores = calc(seed, data_selected, features, target)
    print(f"Seed: {seed}, RMSE: {rmse_scores[0]}")

# Compute the standard deviation of the RMSE scores
import numpy as np
std = np.std(rmse_scores)
print(f"Standard deviation of RMSE scores: {std}")


Seed: 0, RMSE: 2.1662408533301045
Seed: 1, RMSE: 2.136081240603674
Seed: 2, RMSE: 2.242072089629972
Seed: 3, RMSE: 2.195525625386097
Seed: 4, RMSE: 2.2002423913899727
Seed: 5, RMSE: 2.204502548217246
Seed: 6, RMSE: 2.150889685779692
Seed: 7, RMSE: 2.20810271129423
Seed: 8, RMSE: 2.214840440384025
Seed: 9, RMSE: 2.2268231648790673
Standard deviation of RMSE scores: 0.0


# Question 6
- Split the dataset like previously, use seed 9.
- Combine train and validation datasets.
- Fill the missing values with 0 and train a model with r=0.001.
- What's the RMSE on the test dataset?


In [72]:
def calc(seed,data_selected,features,target):
    rmse_scores = []
    data_selected_train, data_selected_test = train_test_split(data_selected, test_size=0.2, random_state=seed )
    data_selected_train_filled = data_selected_train.fillna(0)
    data_selected_test_filled= data_selected_test.fillna(0)
    model =Ridge(alpha=0.001)
    model.fit(data_selected_train_filled[features], data_selected_train_filled[target])
    y_predict = model.predict(data_selected_test_filled[features])
    rmse = root_mean_squared_error(data_selected_test_filled[target], y_predict)
    rmse_scores.append(rmse)
    return rmse_scores


In [73]:
 rmse_scores = calc(9,data_selected, features, target)
 rmse_scores

[2.2253500365813244]